# Deploy a Gemma 4 Model on OpenShift AI

This notebook guides you through deploying a **Gemma 4** model on OpenShift AI using a custom vLLM ServingRuntime. Once deployed, you can evaluate the model using the LMEvalJob notebooks in this workshop.

## Overview

We will:
1. Register a custom vLLM ServingRuntime with Gemma 4 support
2. Deploy the model as a KServe InferenceService
3. Verify the deployment

## Prerequisites

- OpenShift AI cluster with GPU nodes (NVIDIA)
- `oc` CLI logged in with admin or project-level privileges
- Hugging Face access to `google/gemma-4-E2B-it` (accept license on HF)

## Step 1: Configuration

In [ ]:
NAMESPACE = "hyo-project"
MODEL_NAME = "vllm-gemma4-e2b"
RUNTIME_NAME = "vllm-cuda-runtime-gemma4"
HF_MODEL_ID = "google/gemma-4-E2B-it"

## Step 2: Register Custom ServingRuntime

OpenShift AI ships with default ServingRuntimes, but Gemma 4 requires a preview vLLM image with Gemma 4 support. We register a custom runtime that uses `registry.redhat.io/rhaii-preview/vllm-cuda-rhel9:gemma4`.

You can also do this via the OpenShift AI Dashboard:
- Navigate to **Settings > Serving runtimes > Add serving runtime**
- Paste the YAML below

![Custom ServingRuntime Registration](../images/serving-runtime-registration.png)

In [ ]:
!cat samples/serving-runtime-gemma4.yaml

In [ ]:
!oc apply -f samples/serving-runtime-gemma4.yaml -n {NAMESPACE}

## Step 3: Create HF Token Secret for Model Download

Gemma 4 is a gated model on Hugging Face. The ServingRuntime needs access to download it:

In [ ]:
# Skip if you already created hf-token in 1_setup.ipynb
!oc get secret hf-token -n {NAMESPACE} 2>/dev/null || \
    echo "Run 1_setup.ipynb first to create hf-token secret"

## Step 4: Deploy the InferenceService

This creates a KServe InferenceService that:
- Uses the custom Gemma 4 runtime
- Pulls the model from Hugging Face (`hf://google/gemma-4-E2B-it`)
- Enables OAuth authentication
- Requests 1 NVIDIA GPU

You can also deploy via the OpenShift AI Dashboard:
- Navigate to **Model Serving > Deploy model**
- Select the custom runtime and configure the model

![Model Deployment](../images/model-deploy.png)

In [ ]:
!cat samples/inferenceservice-gemma4.yaml

In [ ]:
!oc apply -f samples/inferenceservice-gemma4.yaml -n {NAMESPACE}

## Step 5: Wait for Model to be Ready

The model download and initialization takes a few minutes (typically 3-5 min for Gemma 4 E2B):

In [ ]:
import time

for i in range(20):
    !oc get inferenceservice {MODEL_NAME} -n {NAMESPACE} -o jsonpath='{.status.conditions[?(@.type=="Ready")].status}'
    print(f"  ({(i+1)*15}s elapsed)")
    time.sleep(15)

## Step 6: Verify Deployment

In [ ]:
!oc get inferenceservice -n {NAMESPACE}
print("\n--- Deployment pods ---")
!oc get pods -n {NAMESPACE} | grep {MODEL_NAME}

## Step 7: Confirm served_model_name

The `served_model_name` is automatically set to the InferenceService name. Verify in vLLM logs:

In [ ]:
!oc logs deployment/{MODEL_NAME}-predictor -n {NAMESPACE} 2>&1 | grep served_model_name | head -1

## Key Configuration Notes

| Setting | Value | Purpose |
|---------|-------|--------|
| `served_model_name` | `{{.Name}}` (InferenceService name) | Used as `model` in LMEvalJob |
| OAuth auth | `security.opendatahub.io/enable-auth: "true"` | Requires SA token for API access |
| Port | `8443` (OAuth proxy) / `8080` (vLLM) | LMEvalJob connects to 8443 |
| Internal URL | `https://<name>-predictor.<ns>.svc.cluster.local:8443` | Used as `base_url` |

## Done!

Your Gemma 4 model is now deployed and ready for evaluation. Proceed to:
- **1_setup.ipynb** — Configure RBAC and secrets for LMEvalJob
- **1_builtin_tasks/** — Run evaluations with built-in tasks
- **2_custom_tasks/** — Run full Korean benchmarks